In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [13]:
model = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

In [14]:
# create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [16]:
def llm_qa(state: LLMState) -> LLMState:
    question = state["question"]
    
    prompt = f"Answer the following question: {question}"
    answer = model.invoke(prompt)
    state["answer"] = answer.content
    
    return state
    

In [17]:
# create our graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [18]:
initial_state = LLMState(question="What is the capital of France?", answer="")

# run the workflow
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'What is the capital of France?', 'answer': [{'type': 'text', 'text': 'The capital of France is **Paris**.', 'extras': {'signature': 'Eq0CCqoCAWkUfRNlB+fAvtOOIuPgZx3pT1kzKFDD9ulagOuFgHmoLwR3tZGGtSj5brPR0BgF+RAalsdaj5cpveswexZt0Wmp7JzBAtbWBw+R3M/oc0SJrjBvFS4p8BRb20LTPiQNUqQzC/Ovem0LXYOB62w92ffYdpX8z+8ekh8NgdM6W3PrPjDd5LLEgTQOVPmHbh+GeWHiORFrw5FbWQfF/1hNPQHVve11QJSS8N4jeXcR7UeHUjv988u2EHGhUlTMU1aMHkgKYgTZZl3IxjLRWez3WdbVjOoYgnM9GvnIUm3tQG3y5ZLY7Bye/VQJYB138PbI5NDKuhZFcBx7cvLhhA5n3aRbKXa+e5ZdSQHZ1WuYRbXWemczzcMAT0EowqaoWVA6ynSda5uAsD/WKg=='}}]}
